In [1]:
# configuración: imports y carga de archivos
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import KNeighborsClassifier
from sklearn.covariance import EllipticEnvelope
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve, roc_auc_score
import warnings; warnings.filterwarnings("ignore")

SEEDS         = list(range(10))
MARKOV_ORDERS = [0, 1, 2, 3, 4]
WINDOW_SIZE   = 50

COLUMN_NAMES = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

RUTA = (Path.home() / ".cache" / "kagglehub" / "datasets" / "hassan06" / "nslkdd" / "versions" / "1" / "KDDTrain+_20Percent.txt")
if not RUTA.exists():
    found = sorted(Path.home().glob("**/*KDDTrain*20Percent*.txt"))
    if found:
        RUTA = found[0]

df = pd.read_csv(RUTA, header=None, names=COLUMN_NAMES).drop(columns=['difficulty'])
print(f"Dataset  : {RUTA.name}")
print(f"Shape    : {df.shape}")
print(f"Etiquetas:\n{df['label'].value_counts().to_string()}")


Dataset  : KDDTrain+_20Percent.txt
Shape    : (25192, 42)
Etiquetas:
label
normal             13449
neptune             8282
ipsweep              710
satan                691
portsweep            587
smurf                529
nmap                 301
back                 196
teardrop             188
warezclient          181
pod                   38
guess_passwd          10
warezmaster            7
buffer_overflow        6
imap                   5
rootkit                4
multihop               2
phf                    2
ftp_write              1
land                   1
loadmodule             1
spy                    1


In [2]:
# preprocesamiento (igual que VOTING ENSAMBLE NSL-KDD)

# Estado Markoviano: service en string original (antes de encoding)
estados = df['service'].astype(str).values

# Etiquetas 5 clases (igual que VE NSL-KDD y Clasificadores NSL-KDD)
DOS   = {'back','land','neptune','pod','smurf','teardrop','apache2','udpstorm','processtable','mailbomb'}
PROBE = {'ipsweep','nmap','portsweep','satan','mscan','saint'}
R2L   = {'ftp_write','guess_passwd','imap','multihop','phf','spy','warezclient','warezmaster',
         'sendmail','named','snmpgetattack','snmpguess','xlock','xsnoop','worm'}
U2R   = {'buffer_overflow','loadmodule','perl','rootkit','httptunnel','ps','sqlattack','xterm'}

def map_label(label):
    label = label.lower().strip()
    if label == 'normal': return 1
    if label in DOS:      return 2
    if label in PROBE:    return 3
    if label in R2L:      return 4
    if label in U2R:      return 5
    return 2

y_multi = np.array([map_label(l) for l in df['label']], dtype=np.int8)

# Encoding igual que VOTING ENSAMBLE NSL-KDD
protocol_map = {'tcp': 1, 'udp': 2, 'icmp': 3}
df['protocol_type'] = df['protocol_type'].str.lower().map(protocol_map).fillna(0)
df['service'] = LabelEncoder().fit_transform(df['service'].astype(str))
df['flag']    = LabelEncoder().fit_transform(df['flag'].astype(str))

# 8 features seleccionadas por Information Gain > 0.40 (igual que VE NSL-KDD)
SELECTED_FEATURES = [
    'src_bytes', 'service', 'dst_bytes', 'flag',
    'diff_srv_rate', 'same_srv_rate', 'dst_host_srv_count', 'dst_host_same_srv_rate',
]
FEAT_COLS = SELECTED_FEATURES

X = df[SELECTED_FEATURES].values.astype(np.float32)

# Etiqueta binaria para deteccion de anomalias (Normal=0, Ataque=1)
y_bin = (y_multi != 1).astype(int)

# CORREGIDO: el scaler se aplica dentro del bucle de semillas (post-split)
# X queda sin escalar; MinMaxScaler se ajusta solo sobre train en Bloque 3
X_full = X  # alias temporal; se reemplaza por arrays escalados dentro del bucle

print(f'Features usadas ({len(FEAT_COLS)}): {FEAT_COLS}')
print(f'Normal (0): {(y_bin==0).sum():,}  |  Ataque (1): {(y_bin==1).sum():,}')
print(f'Tasa de anomalias: {y_bin.mean():.1%}')
print(f'\nEstados unicos: {len(np.unique(estados))}')
print(pd.Series(estados).value_counts().head(10).to_string())


Features usadas (8): ['src_bytes', 'service', 'dst_bytes', 'flag', 'diff_srv_rate', 'same_srv_rate', 'dst_host_srv_count', 'dst_host_same_srv_rate']
Normal (0): 13,449  |  Ataque (1): 11,743
Tasa de anomalias: 46.6%

Estados unicos: 66
http        8003
private     4351
domain_u    1820
smtp        1449
ftp_data    1396
eco_i        909
other        858
ecr_i        613
telnet       483
finger       366


In [ ]:
def build_sequences_windowed(states_array, window_size):
    """Divide el array de estados en ventanas consecutivas de tamaño fijo."""
    seqs, orig_idxs = [], []
    n = len(states_array)
    for start in range(0, n, window_size):
        end = min(start + window_size, n)
        seqs.append(list(states_array[start:end]))
        orig_idxs.append(list(range(start, end)))
    return seqs, orig_idxs

def build_tm(sequences, order):
    """Matriz de transición de orden k aprendida sobre secuencias de entrenamiento."""
    counts = defaultdict(lambda: defaultdict(int))
    for seq in sequences:
        for t in range(order, len(seq)):
            counts[tuple(seq[t - order:t])][seq[t]] += 1
    return {ctx: {s: n / sum(v.values()) for s, n in v.items()}
            for ctx, v in counts.items()}

def compute_scores(sequences, tm, order, eps=1e-10):
    """Surprise, entropy y log-likelihood por observación (Ghazi et al., Eq. 1-3)."""
    out = []
    for seq in sequences:
        for t, state in enumerate(seq):
            if t < order:
                out.append((0., 0., 0.)); continue
            ctx   = tuple(seq[t - order:t])
            trans = tm.get(ctx, {})
            p     = trans.get(state, eps)
            dist  = np.array(list(trans.values())) if trans else np.array([eps])
            dist /= dist.sum()
            out.append((
                -np.log(p + eps),
                -np.sum(dist * np.log(dist + eps)),
                np.log(p + eps)
            ))
    return np.array(out)

def markov_features(estados, idx_tr, idx_ev, order):
    """Devuelve DataFrame con 3 features Markovianas alineadas con idx_ev."""
    seqs_tr, _ = build_sequences_windowed(estados[idx_tr], WINDOW_SIZE)
    tm          = build_tm(seqs_tr, order)
    seqs_ev, _ = build_sequences_windowed(estados[idx_ev], WINDOW_SIZE)
    scores      = compute_scores(seqs_ev, tm, order)
    return pd.DataFrame({"surprise": scores[:, 0], "entropy": scores[:, 1], "loglik": scores[:, 2]})

def youden_thr(y_true, sc):
    fpr, tpr, thr = roc_curve(y_true, sc)
    return thr[np.argmax(tpr - fpr)]

def get_models(seed):
    return {
        "RandomForest"   : RandomForestClassifier(n_estimators=50, random_state=seed, n_jobs=-1),
        "MLP"            : MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=50,
                                         early_stopping=True, n_iter_no_change=5,
                                         tol=1e-3, random_state=seed),
        "KNN"            : KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "IsolationForest": IsolationForest(n_estimators=100, contamination="auto",
                                           random_state=seed, n_jobs=-1),
        "OCSVM"          : OneClassSVM(kernel="rbf", nu=0.1, gamma="scale"),
        "RobustCov"      : EllipticEnvelope(contamination=0.3, random_state=seed),
    }

UNSUP_SUB = {"OCSVM", "RobustCov"}

all_results = []
t0_total = time.time()

for seed in SEEDS:
    t0_seed = time.time()

    # Split estratificado 64 / 16 / 20 (test=20%, igual que VE NSL-KDD)
    X_tv, X_te, y_tv, y_te, idx_tv, idx_te = train_test_split(
        X, y_bin, np.arange(len(y_bin)), test_size=0.20, random_state=seed, stratify=y_bin
    )
    X_tr, X_vl, y_tr, y_vl, idx_tr, idx_vl = train_test_split(
        X_tv, y_tv, idx_tv, test_size=0.20, random_state=seed, stratify=y_tv
    )

    # CORREGIDO: scaler ajustado solo sobre train de cada semilla (post-split)
    scaler_s = MinMaxScaler()
    X_tr_s = pd.DataFrame(scaler_s.fit_transform(X_tr), columns=FEAT_COLS)
    X_vl_s = pd.DataFrame(scaler_s.transform(X_vl),     columns=FEAT_COLS)
    X_te_s = pd.DataFrame(scaler_s.transform(X_te),     columns=FEAT_COLS)

    for k in MARKOV_ORDERS:
        t0_k = time.time()

        if k == 0:
            Xtr, Xvl, Xte = X_tr_s, X_vl_s, X_te_s
        else:
            mk_tr = markov_features(estados, idx_tr, idx_tr, k)
            mk_vl = markov_features(estados, idx_tr, idx_vl, k)
            mk_te = markov_features(estados, idx_tr, idx_te, k)
            Xtr = pd.concat([X_tr_s.reset_index(drop=True), mk_tr], axis=1)
            Xvl = pd.concat([X_vl_s.reset_index(drop=True), mk_vl], axis=1)
            Xte = pd.concat([X_te_s.reset_index(drop=True), mk_te], axis=1)

        Xtr_norm = Xtr[y_tr == 0]
        rng      = np.random.default_rng(seed)
        sub_idx  = rng.choice(len(Xtr_norm), size=min(5000, len(Xtr_norm)), replace=False)
        Xtr_sub  = Xtr_norm.iloc[sub_idx]

        for name, model in get_models(seed).items():
            t0_m = time.time()

            if name in ("RandomForest", "MLP", "KNN"):
                model.fit(Xtr, y_tr)
                sc_vl = model.predict_proba(Xvl)[:, 1]
                sc_te = model.predict_proba(Xte)[:, 1]
            else:
                fit_data = Xtr_sub if name in UNSUP_SUB else Xtr_norm
                model.fit(fit_data)
                score_fn = model.score_samples if hasattr(model, "score_samples") else model.decision_function
                sc_vl = -score_fn(Xvl)
                sc_te = -score_fn(Xte)

            pred = (sc_te >= youden_thr(y_vl, sc_vl)).astype(int)
            all_results.append({
                "seed" : seed,
                "k"    : k,
                "model": name,
                "AUC"  : roc_auc_score(y_te, sc_te),
                "F1"   : f1_score(y_te, pred, zero_division=0),
                "Prec" : precision_score(y_te, pred, zero_division=0),
                "Rec"  : recall_score(y_te, pred, zero_division=0),
            })
            print(f"  seed={seed} k={k} {name:<18} {time.time()-t0_m:5.1f}s", flush=True)

        print(f"  --- k={k} completado en {time.time()-t0_k:.1f}s ---", flush=True)

    elapsed   = time.time() - t0_seed
    remaining = elapsed * (len(SEEDS) - seed - 1)
    print(f"Semilla {seed} completada en {elapsed:.0f}s | ETA restante: {remaining/60:.1f} min\n", flush=True)


  seed=0 k=0 RandomForest         0.2s


  seed=0 k=0 MLP                  1.5s


  seed=0 k=0 KNN                  0.4s


  seed=0 k=0 IsolationForest      0.4s


  seed=0 k=0 OCSVM                1.2s


...

  seed=9 k=4 IsolationForest      0.4s


  seed=9 k=4 OCSVM                0.9s


  seed=9 k=4 RobustCov            1.0s


  --- k=4 completado en 7.1s ---


Semilla 9 completada en 33s | ETA restante: 0.0 min



In [4]:
# resultados: métricas sobre el set de test

df_res = pd.DataFrame(all_results)
names  = list(df_res["model"].unique())

W     = 102
TITLE = "TABLA DE RESULTADOS — AUC y F1-Score por modelo y orden Markoviano"
HDR   = (f"{'Modelo':<18} {'Orden':>5}   {'AUC media':>10}  {'±std':>6}   "
         f"{'F1 media':>9}  {'±std':>6}   {'Precisión':>9}  {'Recall':>7}   Tend.")
SEP   = "-" * W

print("=" * W)
print(TITLE.center(W))
print("=" * W)
print(HDR)
print(SEP)

for i, name in enumerate(names):
    aucs = [df_res[(df_res["model"] == name) & (df_res["k"] == k)]["AUC"].mean()
            for k in MARKOV_ORDERS]
    d = aucs[-1] - aucs[0]

    if   max(aucs) - min(aucs) < 0.001:                              tend = "estable"
    elif all(aucs[j] >= aucs[j-1] for j in range(1, len(aucs))):     tend = f"mejora  d=+{d:.4f}"
    elif all(aucs[j] <= aucs[j-1] for j in range(1, len(aucs))):     tend = f"degrada d={d:.4f}"
    else:                                                             tend = f"mixta   d={d:+.4f}"

    for k in MARKOV_ORDERS:
        sub  = df_res[(df_res["model"] == name) & (df_res["k"] == k)]
        r    = {m: sub[m].mean() for m in ("AUC", "F1", "Prec", "Rec")}
        rs   = {m: sub[m].std()  for m in ("AUC", "F1")}

        if   k == 0:                            arrow = "-"
        elif aucs[k] > aucs[k-1] + 0.0005:     arrow = "sube"
        elif aucs[k] < aucs[k-1] - 0.0005:     arrow = "baja"
        else:                                   arrow = "="

        tend_col = tend if k == 0 else ""
        print(f"{name:<18} {'k='+str(k):>5}   {r['AUC']:>10.4f}  {rs['AUC']:>6.4f}   "
              f"{r['F1']:>9.4f}  {rs['F1']:>6.4f}   "
              f"{r['Prec']:>9.4f}  {r['Rec']:>7.4f}   {arrow:<5}  {tend_col}")

    if i < len(names) - 1:
        print()
        print(SEP)

print("=" * W)

print()
print("=" * 72)
print(f"{'RESUMEN DE TENDENCIAS Y VEREDICTO':^72}")
print("=" * 72)
for name in names:
    aucs   = [df_res[(df_res["model"] == name) & (df_res["k"] == k)]["AUC"].mean()
              for k in MARKOV_ORDERS]
    f1s    = [df_res[(df_res["model"] == name) & (df_res["k"] == k)]["F1"].mean()
              for k in MARKOV_ORDERS]
    best_k = int(np.argmax(aucs))
    d_auc  = aucs[-1] - aucs[0]
    print(f"\n  {name}")
    print("    AUC por orden: " + "  ".join(f"k={k}:{aucs[k]:.4f}" for k in MARKOV_ORDERS))
    print("    F1  por orden: " + "  ".join(f"k={k}:{f1s[k]:.4f}"  for k in MARKOV_ORDERS))
    print(f"    Mejor k       : k={best_k}  AUC={aucs[best_k]:.4f}  F1={f1s[best_k]:.4f}")
    if   d_auc > 0.002:   verdict = "Markov AYUDA"
    elif d_auc < -0.002:  verdict = "Markov PERJUDICA"
    else:                 verdict = "Markov SIN EFECTO CLARO"
    print(f"    Veredicto     : {verdict}  (delta AUC k0→k4 = {d_auc:+.4f})")   


                  TABLA DE RESULTADOS — AUC y F1-Score por modelo y orden Markoviano                  
Modelo             Orden    AUC media    ±std    F1 media    ±std   Precisión   Recall   Tend.
------------------------------------------------------------------------------------------------------
RandomForest         k=0       0.9995  0.0003      0.9933  0.0022      0.9917   0.9949   -      estable
RandomForest         k=1       0.9993  0.0005      0.9922  0.0026      0.9913   0.9931   =      
RandomForest         k=2       0.9992  0.0005      0.9902  0.0026      0.9890   0.9913   =      
RandomForest         k=3       0.9990  0.0005      0.9905  0.0033      0.9881   0.9928   =      
RandomForest         k=4       0.9990  0.0007      0.9919  0.0029      0.9905   0.9933   =      

------------------------------------------------------------------------------------------------------
MLP                  k=0       0.9716  0.0030      0.9146  0.0064      0.9650   0.8693   -      mixta  